## Test if CUDA is installed with PyTorch


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

## Create directories and set filenames

In [ ]:
import os 



video_path = "data/video.MOV"
image_dir = "data/frames"
workspace = "data/workspace"
database = workspace + "database.db"
sparse = workspace + "/sparse"
dense = workspace + "/dense"
ply_output = workspace + "/pointcloud.ply"
#maximagesize is low here because my GPU is old
max_image_size = 500

os.makedirs(sparse, exist_ok=True)
os.makedirs(dense, exist_ok=True)

## Extract Frames from Video

In [ ]:
import cv2
import os

#not every frame of the video has to be used 
step = 5

os.makedirs(image_dir, exist_ok=True)

cap = cv2.VideoCapture(video_path)

i = 0
frame_id = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if i % step == 0:
        cv2.imwrite(f"{image_dir}/{frame_id:05d}.jpg", frame)
        frame_id += 1

    i += 1

cap.release()
print(f"Extracted {frame_id} frames")

## Extract Features 

In [78]:
!colmap feature_extractor \
 --database_path {database} \
 --image_path {image_dir} \
 --ImageReader.single_camera 1 \
 --SiftExtraction.max_image_size {max_image_size}


Feature extraction

Processed file [1/80]
  Name:            00000.jpg
  SKIP: Features for image already extracted.
Processed file [2/80]
  Name:            00001.jpg
  SKIP: Features for image already extracted.
Processed file [3/80]
  Name:            00002.jpg
  SKIP: Features for image already extracted.
Processed file [4/80]
  Name:            00003.jpg
  SKIP: Features for image already extracted.
Processed file [5/80]
  Name:            00004.jpg
  SKIP: Features for image already extracted.
Processed file [6/80]
  Name:            00005.jpg
  SKIP: Features for image already extracted.
Processed file [7/80]
  Name:            00006.jpg
  SKIP: Features for image already extracted.
Processed file [8/80]
  Name:            00007.jpg
  SKIP: Features for image already extracted.
Processed file [9/80]
  Name:            00008.jpg
  SKIP: Features for image already extracted.
Processed file [10/80]
  Name:            00009.jpg
  SKIP: Features for image already extracted.
Processe

## Match Features

In [79]:
!colmap exhaustive_matcher \
  --database_path {database}


Exhaustive feature matching

Matching block [1/2, 1/2] in 309.042s
Matching block [1/2, 2/2] in 107.406s
Matching block [2/2, 1/2] in 248.366s
Matching block [2/2, 2/2] in 93.604s
Elapsed time: 12.660 [minutes]


## Mapper Step (build sparse reconstruction)

In [80]:
!colmap mapper   \
--database_path {database}  \
 --image_path {image_dir} \
   --output_path {sparse}


Loading database

Loading cameras... 1 in 0.000s
Loading matches... 3130 in 0.722s
Loading images... 80 in 0.315s (connected 80)
Building correspondence graph... in 5.029s (ignored 0)

Elapsed time: 0.102 [minutes]


Finding good initial image pair


Initializing with image pair #46 and #52


Global bundle adjustment

iter      cost      cost_change  |gradient|   |step|    tr_ratio  tr_radius  ls_iter  iter_time  total_time
   0  3.956258e+02    0.00e+00    1.38e+04   0.00e+00   0.00e+00  1.00e+04        0    1.55e-02    4.95e-02
   1  3.668860e+02    2.87e+01    4.25e+03   4.51e+00   9.98e-01  3.00e+04        1    2.66e-02    7.62e-02
   2  3.660242e+02    8.62e-01    1.46e+03   5.07e+00   9.59e-01  9.00e+04        1    9.90e-02    1.75e-01
   3  3.658219e+02    2.02e-01    1.20e+03   4.39e+00   9.08e-01  1.97e+05        1    2.68e-02    2.02e-01
   4  3.657522e+02    6.97e-02    7.54e+02   3.14e+00   9.07e-01  4.28e+05        1    6.10e-02    2.63e-01
   5  3.657213e+02    3.09e-02 

## Undistort images 

In [84]:
!colmap image_undistorter \
--image_path {image_dir}  \
--input_path {sparse}/0  \
--output_path {dense}  \
--output_type COLMAP \
--max_image_size {max_image_size}


Reading reconstruction

 => Reconstruction with 80 images and 56619 points

Image undistortion

Undistorting image [1/80]
Undistorting image [2/80]
Undistorting image [3/80]
Undistorting image [4/80]
Undistorting image [5/80]
Undistorting image [6/80]
Undistorting image [7/80]
Undistorting image [8/80]
Undistorting image [9/80]
Undistorting image [10/80]
Undistorting image [11/80]
Undistorting image [12/80]
Undistorting image [13/80]
Undistorting image [14/80]
Undistorting image [15/80]
Undistorting image [16/80]
Undistorting image [17/80]
Undistorting image [18/80]
Undistorting image [19/80]
Undistorting image [20/80]
Undistorting image [21/80]
Undistorting image [22/80]
Undistorting image [23/80]
Undistorting image [24/80]
Undistorting image [25/80]
Undistorting image [26/80]
Undistorting image [27/80]
Undistorting image [28/80]
Undistorting image [29/80]
Undistorting image [30/80]
Undistorting image [31/80]
Undistorting image [32/80]
Undistorting image [33/80]
Undistorting image [3

In [85]:
!colmap patch_match_stereo  \
--workspace_path {dense} \
--workspace_format COLMAP \
--PatchMatchStereo.geom_consistency true \
--PatchMatchStereo.max_image_size {max_image_size}


Reading workspace...
Reading configuration...
Configuration has 80 problems...

Processing view 1 / 80 for 00000.jpg

Reading inputs...

PatchMatch::Problem
-------------------
ref_image_idx: 0
src_image_idxs: 4 6 8 5 7 9 10 11 12 13 14 15 16 17 19 18 20 21 22 23

PatchMatchOptions
-----------------
max_image_size: 500
gpu_index: 0
depth_min: 5.52917
depth_max: 17.0512
window_radius: 5
window_step: 1
sigma_spatial: 5
sigma_color: 0.2
num_samples: 15
ncc_sigma: 0.6
min_triangulation_angle: 1
incident_angle_sigma: 0.9
num_iterations: 5
geom_consistency: 0
geom_consistency_regularizer: 0.3
geom_consistency_max_cost: 3
filter: 0
filter_min_ncc: 0.1
filter_min_triangulation_angle: 3
filter_min_num_consistent: 2
filter_geom_consistency_max_cost: 1
write_consistency_graph: 0
allow_missing_files: 0

PatchMatch::Run
---------------
Initialization: 0.0794s
 Sweep 1: 0.4081s
 Sweep 2: 0.3896s
 Sweep 3: 0.3950s
 Sweep 4: 0.3861s
Iteration 1: 1.5795s
 Sweep 1: 0.3783s
 Sweep 2: 0.3754s
 Sweep 3: 0.

In [99]:
!colmap stereo_fusion \
 --workspace_path {dense} \
 --workspace_format COLMAP \
 --input_type geometric \
 --output_path {ply_output} 


StereoFusion::Options
---------------------
mask_path: 
max_image_size: -1
min_num_pixels: 5
max_num_pixels: 10000
max_traversal_depth: 100
max_reproj_error: 2
max_depth_error: 0.01
max_normal_error: 10
check_num_images: 50
use_cache: 0
cache_size: 32
bbox_min: -3.40282e+38 -3.40282e+38 -3.40282e+38
bbox_max: 3.40282e+38 3.40282e+38 3.40282e+38

Reading workspace...
Loading workspace data with 4 threads...
Elapsed time: 0.020 [minutes]
Reading configuration...
Starting fusion with 4 threads
Fusing image [1/80] with index 0 in 7.417s (31430 points)
Fusing image [2/80] with index 1 in 1.095s (39898 points)
Fusing image [3/80] with index 2 in 0.717s (45679 points)
Fusing image [4/80] with index 4 in 0.886s (52379 points)
Fusing image [5/80] with index 3 in 0.368s (55238 points)
Fusing image [6/80] with index 6 in 0.670s (60509 points)
Fusing image [7/80] with index 8 in 0.579s (64811 points)
Fusing image [8/80] with index 9 in 0.479s (67935 points)
Fusing image [9/80] with index 10 in 0.

## Visualize PointCloud in Open3d

In [2]:
import open3d as o3d

# Load PLY file
pcd = o3d.io.read_point_cloud(ply_output)

# Optional: print info
print(pcd)

# Visualize
o3d.visualization.draw_geometries([pcd])

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
PointCloud with 211299 points.
